In [7]:
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator

# =====================================================================
# STEP 2: Create a Three-Qubit Circuit (with 3 classical bits)
# Qubit 0 -> Alice's unknown state
# Qubit 1 -> Alice's half of the Bell pair
# Qubit 2 -> Bob's half of the Bell pair
# Classical Bits 0 & 1 -> Alice's measurement data
# Classical Bit 2 -> Bob's final state verification
# =====================================================================
qc = QuantumCircuit(3, 3)

# =====================================================================
# STEP 3: Prepare the Unknown State to Teleport
# Putting Qubit 0 into the |+⟩ state as a test case
# =====================================================================
qc.h(0)
qc.barrier() # Keeps sections separated visually

# =====================================================================
# STEP 4: Create the Shared Bell Pair
# Entangling Qubit 1 and Qubit 2 to establish the quantum bridge
# =====================================================================
qc.h(1)
qc.cx(1, 2)
qc.barrier()

# =====================================================================
# STEP 5: Alice's Operations
# Interacting her unknown qubit with her half of the entangled pair
# =====================================================================
qc.cx(0, 1)
qc.h(0)
qc.barrier()

# =====================================================================
# STEP 6: Measure Alice's Qubits
# Collapsing her quantum states into normal classical data bits
# =====================================================================
qc.measure(0, 0) # Qubit 0 maps to Classical Bit 0
qc.measure(1, 1) # Qubit 1 maps to Classical Bit 1
qc.barrier()

# =====================================================================
# STEP 7: Bob's Conditional Corrections
# Bob looks at Alice's classical bits to fix his qubit (Qubit 2)
# =====================================================================
with qc.if_test((0, 1)):  # If Alice's Bit 0 is 1 -> apply X gate
    qc.x(2)

with qc.if_test((1, 1)):  # If Alice's Bit 1 is 1 -> apply Z gate
    qc.z(2)
qc.barrier()

# =====================================================================
# STEP 8: Verify Bob Recovers the State
# Measure Bob's final qubit to read out his state vector collapse
# =====================================================================
qc.measure(2, 2) # Qubit 2 maps to Classical Bit 2

# =====================================================================
# EXECUTION: Simulate the Complete Circuit
# =====================================================================
sim = AerSimulator()
compiled_circuit = transpile(qc, sim)
result = sim.run(compiled_circuit, shots=1024).result()

# Print out the dictionary of final simulated states
print("Output format: [Bob's Bit] [Alice's Bit 1] [Alice's Bit 0]")
print(result.get_counts())

Output format: [Bob's Bit] [Alice's Bit 1] [Alice's Bit 0]
{'001': 129, '101': 125, '100': 135, '011': 145, '111': 127, '110': 122, '010': 123, '000': 118}
